# **Recap of [Lesson 3](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_3_pagination_and_authentication_and_dlt_configuration.ipynb) 👩‍💻🚀**

1. Used pagination with REST APIs.
2. Applied authentication for REST APIs.
3. Tried the dlt `RESTClient`.
4. Used environment variables to manage secrets and configuration.
5. Learned how to add values to `secrets.toml` and `config.toml`.
6. Used the special `secrets.toml` environment variable setup for Colab.

---
# **`dlt`’s pre-built Sources and Destinations** [![Open in molab](https://marimo.io/molab-shield.svg)](https://molab.marimo.io/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_4_using_pre_build_sources_and_destinations.py) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_4_using_pre_build_sources_and_destinations.ipynb) [![GitHub badge](https://img.shields.io/badge/github-view_source-2b3137?logo=github)](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_4_using_pre_build_sources_and_destinations.ipynb)


**Here, you will learn:**
- How to initialize verified sources.
- The built-in `rest_api` source.
- The built-in `sql_database` source.
- The built-in `filesystem` source.
- How to switch between destinations.

---

Our verified sources are the simplest way to start building your stack. Choose from any of our fully customizable 30+ pre-built sources, such as SQL databases, Google Sheets, Salesforce, and more.

With our numerous destinations, you can load data into a local database, data warehouse, or data lake. Choose from Snowflake, Databricks, and many others.

![Lesson_4_Using_pre_build_sources_and_destinations_img1](https://storage.googleapis.com/dlt-blog-images/dlt-fundamentals-course/Lesson_4_Using_pre_build_sources_and_destinations_img1.png)

# **Existing verified sources**
To use an [existing verified source](https://dlthub.com/docs/dlt-ecosystem/verified-sources/), just run the `dlt init` command.





There's a base project for each `dlt` verified source + destination combination, which you can adjust according to your needs.

These base project can be initialized with a simple command:

```
dlt init <verified-source> <destination>
```

In [26]:
%%capture
!pip install dlt[duckdb]

List all verified sources:


In [27]:
!dlt init --list-sources

---
Available dlt core sources:
---
filesystem: Reads files in s3, gs or azure buckets using fsspec and provides convenience resources for chunked reading of various file formats
rest_api: Generic API Source
sql_database: Source that loads tables form any SQLAlchemy supported database, supports batching requests and incremental loads.
---
Available dlt single file templates:
---
arrow: The Arrow Pipeline Template will show how to load and transform arrow tables.
context_rest_api: Template for building a `dlt` pipeline to ingest data from a REST API.
dataframe: The DataFrame Pipeline Template will show how to load and transform pandas dataframes.
debug: The Debug Pipeline Template will load a column with each datatype to your destination.
default: The Intro Pipeline Template contains the example from the docs intro page
fruitshop: The Default Pipeline Template provides a simple starting point for your dlt pipeline with locally generated data
github_api: The Github API templates provides

This command shows all available verified sources and their short descriptions. For each source, it checks if your local `dlt` version requires an update and prints the relevant warning.

Consider an example pipeline for the GitHub API:

```
Available dlt single file templates:
---
arrow: The Arrow Pipeline Template will show how to load and transform arrow tables.
dataframe: The DataFrame Pipeline Template will show how to load and transform pandas dataframes.
debug: The Debug Pipeline Template will load a column with each datatype to your destination.
default: The Intro Pipeline Template contains the example from the docs intro page
fruitshop: The Default Pipeline Template provides a simple starting point for your dlt pipeline

---> github_api: The Github API templates provides a starting

point to read data from REST APIs with REST Client helper
requests: The Requests Pipeline Template provides a simple starting point for a dlt pipeline with the requests library
```

### Step 1. Initialize the source

This command will initialize the pipeline example with the GitHub API as the source and DuckBD as the destination:

In [28]:
!dlt --non-interactive init github_api duckdb

Looking up verified sources at https://github.com/dlt-hub/verified-sources.git...


Now, check  your files on the left side bar. It should contain all the necessary files to run your GitHub API -> DuckDB pipeline:

- The `.dlt` folder containing `secrets.toml` and `config.toml`
- The pipeline script `github_api_pipeline.py`
- `requirements.txt`
- `.gitignore`

In [29]:
!ls -a

.	 .dlt			     .gitignore        rest_api_github.duckdb
..	 github_api_pipeline.duckdb  __pycache__       rest_api_pipeline.py
.config  github_api_pipeline.py      requirements.txt  sample_data


What you would normally do with the project:
- Add your credentials and define configurations
- Adjust the pipeline script as needed
- Run the pipeline script

> If needed, you can adjust the verified source code.

In [30]:
!cat github_api_pipeline.py

"""The Github API templates provides a starting point to read data from REST APIs with REST Client"""

# mypy: disable-error-code="no-untyped-def,arg-type"

from typing import Optional
import dlt
from dlt.sources.helpers.rest_client import RESTClient
from dlt.sources.helpers.rest_client.auth import BearerTokenAuth
from dlt.sources.helpers.rest_client.paginators import HeaderLinkPaginator


@dlt.source
def github_source(access_token: Optional[str] = dlt.secrets.value):
    # Optional auth: works with and without secrets
    auth = BearerTokenAuth(token=access_token) if access_token else None

    client = RESTClient(
        base_url="https://api.github.com",
        auth=auth,
        paginator=HeaderLinkPaginator(),
        headers={
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )

    # Define a resource which fetches repos data
    @dlt.resource(name="repos", write_disposition="replace")
    def repos():
       

From the code, we can see that this pipeline loads **only the `"issues"` endpoint**.
You can adjust this code as needed: add new endpoints, include additional logic, apply transformations, and more.

### Step 2. Add credentials

In Colab (or Molab), it is more convenient to use environment variables or `dlt.secrets`.

In the pipeline above, the `access_token` parameter is set to `dlt.secrets.value`, which means you need to configure this variable:


```python
@dlt.resource(write_disposition="replace")
def github_api_resource(access_token: Optional[str] = dlt.secrets.value):
  ...
```

In [31]:
import os

import dlt
from google.colab import userdata

os.environ["SOURCES__ACCESS_TOKEN"] = userdata.get("SECRET_KEY")

### Step 3. Run the pipeline

Let's run the pipeline!

In [32]:
!python github_api_pipeline.py

---------------------------- Extract github_source -----------------------------
Resources: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 179.74 MB (12.30%) | CPU usage: 0.00%

---------------------------- Extract github_source -----------------------------
Resources: 0/2 (0.0%) | Time: 1.46s | Rate: 0.00/s
repos: 60  | Time: 0.82s | Rate: 72.83/s
Memory usage: 181.18 MB (12.30%) | CPU usage: 0.00%

---------------------------- Extract github_source -----------------------------
Resources: 2/2 (100.0%) | Time: 2.01s | Rate: 0.99/s
repos: 87  | Time: 1.37s | Rate: 63.36/s
Memory usage: 181.48 MB (12.30%) | CPU usage: 0.00%

---------------- Normalize github_source in 1795189565.1711206 -----------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 182.02 MB (12.30%) | CPU usage: 0.00%

2026-11-20 15:46:07,490|[WARNING]|4810|138075518066816|dlt|validate.py|verify_normalized_table:113|In schema `github_source`: The following columns in table 'repos' did not receive

From the pipeline output, we can get information such as the pipeline name, dataset name, destination path, and more.

> Pipeline **github_api_pipeline** load step completed in 1.23 seconds
> 1 load package was loaded to the DuckDB destination and into the dataset **github_api_data**.
> The DuckDB destination used `duckdb:////content/**github_api_pipeline.duckdb**` as the storage location.
> Load package `1733848559.8195539` is **LOADED** and contains no failed jobs.



## Step 4: Explore your data

Let's explore what tables were created in the destination.

In [33]:
pipeline = dlt.attach(pipeline_name="github_api_pipeline")
dataset = pipeline.dataset()

# List the tables that were created in the destination
dataset.tables

['repos',
 'issues',
 'issues__labels',
 'issues__assignees',
 'issues__performed_via_github_app__events',
 'repos__topics',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [34]:
dataset.repos.df()

,id,node_id,name,full_name,private,html_url,fork,url,forks_url,keys_url,...,owner__events_url,owner__received_events_url,owner__type,owner__user_view_type,owner__site_admin,_dlt_load_id,_dlt_id,description,homepage,license__url
0,427664318,R_kgDOGX2jvg,rasa_semantic_schema,dlt-hub/rasa_semantic_schema,False,https://github.com/dlt-hub/rasa_semantic_schema,False,https://api.github.com/repos/dlt-hub/rasa_sema...,https://api.github.com/repos/dlt-hub/rasa_sema...,https://api.github.com/repos/dlt-hub/rasa_sema...,...,https://api.github.com/users/dlt-hub/events{/p...,https://api.github.com/users/dlt-hub/received_...,Organization,public,False,1795189565.1711206,w3MtPdY6JPfShQ,None,None,None
1,438622757,R_kgDOGiTaJQ,.github,dlt-hub/.github,False,https://github.com/dlt-hub/.github,False,https://api.github.com/repos/dlt-hub/.github,https://api.github.com/repos/dlt-hub/.github/f...,https://api.github.com/repos/dlt-hub/.github/k...,...,https://api.github.com/users/dlt-hub/events{/p...,https://api.github.com/users/dlt-hub/received_...,Organization,public,False,1795189565.1711206,HfH9eBEGpIhqCg,None,None,None
2,452221115,R_kgDOGvRYuw,dlt,dlt-hub/dlt,False,https://github.com/dlt-hub/dlt,False,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/forks,https://api.github.com/repos/dlt-hub/dlt/keys{...,...,https://api.github.com/users/dlt-hub/events{/p...,https://api.github.com/users/dlt-hub/received_...,Organization,public,False,1795189565.1711206,M/H2/Ua5eJcd0w,data load tool (dlt) is an open source Python ...,https://dlthub.com/docs,https://api.github.com/licenses/apache-2.0
3,465315657,R_kgDOG7wnSQ,metabase_data_api,dlt-hub/metabase_data_api,False,https://github.com/dlt-hub/metabase_data_api,False,https://api.github.com/repos/dlt-hub/metabase_...,https://api.github.com/repos/dlt-hub/metabase_...,https://api.github.com/repos/dlt-hub/metabase_...,...,https://api.github.com/users/dlt-hub/events{/p...,https://api.github.com/users/dlt-hub/received_...,Organization,public,False,1795189565.1711206,hOd6kCtzVbUPnA,Metabase data api python wrapper for notebooks...,,https://api.github.com/licenses/mit
4,526253602,R_kgDOH13-Ig,dlt-axie-infinity-demo,dlt-hub/dlt-axie-infinity-demo,False,https://github.com/dlt-hub/dlt-axie-infinity-demo,False,https://api.github.com/repos/dlt-hub/dlt-axie-...,https://api.github.com/repos/dlt-hub/dlt-axie-...,https://api.github.com/repos/dlt-hub/dlt-axie-...,...,https://api.github.com/users/dlt-hub/events{/p...,https://api.github.com/users/dlt-hub/received_...,Organization,public,False,1795189565.1711206,M4lA8JtD6BQWJw,DLT Pipeline Script to get Axie Infinity smart...,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,1270174547,R_kgDOS7VTUw,dlthub-init,dlt-hub/dlthub-init,False,https://github.com/dlt-hub/dlthub-init,False,https://api.github.com/repos/dlt-hub/dlthub-init,https://api.github.com/repos/dlt-hub/dlthub-in...,https://api.github.com/repos/dlt-hub/dlthub-in...,...,https://api.github.com/users/dlt-hub/events{/p...,https://api.github.com/users/dlt-hub/received_...,Organization,public,False,1795189565.1711206,wUx/xUCTMKbYTQ,Scaffold a new dltHub workspace with built-in ...,None,None
83,1332607030,R_kgDOT234Ng,demo-velodb,dlt-hub/demo-velodb,False,https://github.com/dlt-hub/demo-velodb,False,https://api.github.com/repos/dlt-hub/demo-velodb,https://api.github.com/repos/dlt-hub/demo-velo...,https://api.github.com/repos/dlt-hub/demo-velo...,...,https://api.github.com/users/dlt-hub/events{/p...,https://api.github.com/users/dlt-hub/received_...,Organization,public,False,1795189565.1711206,lItxRT3BnWm7Tg,None,None,None
84,1337244865,R_kgDOT7S8wQ,dlthub-motherduck-lightdash-demo,dlt-hub/dlthub-motherduck-lightdash-demo,False,https://github.com/dlt-hub/dlthub-motherduck-l...,False,https://api.github.com/repos/dlt-hub/dlthub-mo...,https://api.github.com/repos/dlt-hub/dlthub-mo...,https://api.github.com/repos/dlt-hub/dlthub-mo...,...,https://api.githu

# **Built-in sources: RestAPI, SQL database & Filesystem**

## **[RestAPI source](https://dlthub.com/docs/dlt-ecosystem/verified-sources/rest_api/basic)**

`rest_api` is a generic source that lets you create a `dlt` source from any REST API using a declarative configuration. Since most REST APIs follow similar patterns, this source provides a convenient way to define your integration declaratively.

Using a [declarative configuration](https://dlthub.com/docs/dlt-ecosystem/verified-sources/rest_api/basic#source-configuration), you can specify:

- the API endpoints to pull data from,
- their relationships,
- how to handle pagination,
- authentication.

`dlt` handles the rest for you: **unnesting the data, inferring the schema**, and **writing it to the destination**.

In the previous lesson, you already used the REST API Client. `dlt`’s **[RESTClient](https://dlthub.com/docs/general-usage/http/rest-client)** is the **low-level abstraction** that powers the RestAPI source.


### Initialize the `rest_api` template

You can initialize the `rest_api` **template** using the `init` command:


In [35]:
!yes | dlt init rest_api duckdb

In the `rest_api_pipeline.py` script, you will find sources for both the GitHub API and the PokeAPI, defined using the `rest_api` source and `RESTAPIConfig`.

Since the `rest_api` source is a **built-in source**, you don't need to initialize it. You can simply **import** it from `dlt.sources` and start using it.

### Example

Here is a simplified example of how to configure the REST API source to load `issues` and issue `comments` from the GitHub API:


In [36]:
import dlt
from dlt.sources.rest_api import RESTAPIConfig, rest_api_source
from dlt.sources.helpers.rest_client.paginators import PageNumberPaginator

config: RESTAPIConfig = {
    "client": {
        "base_url": "https://api.github.com",
        "auth": {
            "token": dlt.secrets["sources.access_token"],
        },
        "paginator": "header_link",
    },
    "resources": [
        {
            "name": "issues",
            "endpoint": {
                "path": "repos/dlt-hub/dlt/issues",
                "params": {
                    "state": "open",
                },
            },
        },
        {
            "name": "issue_comments",
            "endpoint": {
                "path": "repos/dlt-hub/dlt/issues/{issue_number}/comments",
                "params": {
                    "issue_number": {
                        "type": ("resolve"),
                        "resource": "issues",
                        "field": "number",
                    },
                },
            },
        },
    ],
}

github_source = rest_api_source(config)


rest_api_pipeline = dlt.pipeline(
    pipeline_name="rest_api_github",
    destination="duckdb",
    dataset_name="rest_api_data",
    dev_mode=True,
)

load_info = rest_api_pipeline.run(github_source)
print(load_info)

2026-11-20 15:48:06,206|[WARNING]|1204|137020450766976|dlt|validate.py|verify_normalized_table:113|In schema `rest_api`: The following columns in table 'issue_comments' did not receive any data during this load and therefore could not have their types inferred:
  - minimized
  - performed_via_github_app
  - pin

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'minimized': {'data_type': 'text'}})

2026-11-20 15:48:06,208|[WARNING]|1204|137020450766976|dlt|validate.py|verify_normalized_table:113|In schema `rest_api`: The following columns in table 'issues' did not receive any data during this load and therefore could not have their types inferred:
  - active_lock_reason
  - assignee
  - closed_at
  - closed_by
  - milestone
  - milestone__due_on
  - performed_via_github_app
  - pinned_comment
  - pull_request__me

Pipeline rest_api_github load step finished in 2.72 seconds
1 load package(s) were loaded to destination duckdb and into dataset rest_api_data_20261120034622
The duckdb destination used duckdb:////content/rest_api_github.duckdb location to store data
Load package 1795189582.5052667 is LOADED and contains no failed jobs


In [37]:
rest_api_pipeline.dataset().issues.df()

,url,repository_url,labels_url,comments_url,events_url,html_url,id,node_id,number,title,...,milestone__creator__gists_url,milestone__creator__starred_url,milestone__creator__subscriptions_url,milestone__creator__organizations_url,milestone__creator__repos_url,milestone__creator__events_url,milestone__creator__received_events_url,milestone__creator__type,milestone__creator__user_view_type,milestone__creator__site_admin
0,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/pull/4677,6499707375,PR_kwDOGvRYu88AAAABQhMlcA,4677,DRAFT: docs: build from `devel` instead of `ma...,...,None,None,None,None,None,None,None,None,None,<NA>
1,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/issues/4676,6474656682,I_kwDOGvRYu88AAAABgetrqg,4676,feat(merge): support null-safe IS NOT DISTINCT...,...,None,None,None,None,None,None,None,None,None,<NA>
2,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/issues/4675,6471903327,I_kwDOGvRYu88AAAABgcFoXw,4675,Support delete-insert with destination_scope o...,...,None,None,None,None,None,None,None,None,None,<NA>
3,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/issues/4674,6467368893,I_kwDOGvRYu88AAAABgXw3vQ,4674,filesystem + iceberg: column new in a load is ...,...,None,None,None,None,None,None,None,None,None,<NA>
4,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/issues/4673,6425309608,I_kwDOGvRYu88AAAABfvpxqA,4673,Incremental cursor skips a row committed after...,...,None,None,None,None,None,None,None,None,None,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/issues/986,2146117566,I_kwDOGvRYu85_6ye-,986,HDFS as a destination,...,None,None,None,None,None,None,None,None,None,<NA>
489,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/issues/968,2134047173,I_kwDOGvRYu85_MvnF,968,Add merge key to pipeline,...,None,None,None,None,None,None,None,None,None,<NA>
490,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt/issue...,https://github.com/dlt-hub/dlt/issues/964,2133004455,I_kwDOGvRYu85_IxCn,964,allow to define certain destination capabiliti...,...,None,None,None,None,None,None,None,None,None,<NA>
491,https://api.github.com/repos/dlt-hub/dlt/issue...,https://api.github.com/repos/dlt-hub/dlt,https://api.github

### **Exercise 1: Run `rest_api` source**

Explore the cells above and answer the question below using `sql_client` or `pipeline.dataset()`.

#### **Question**
How many columns does the `issues` table have?

### **Exercise 2: Create a dlt source with `rest_api`**

Add the `contributors` endpoint for the `dlt` repository to the `rest_api` configuration:

- Resource name: **"contributors"**
- Endpoint path: **"repos/dlt-hub/dlt/contributors"**
- No parameters

#### **Question**
How many columns does the `contributors` table have?


In [38]:
import dlt
from dlt.sources.rest_api import RESTAPIConfig, rest_api_source
from dlt.sources.helpers.rest_client.paginators import PageNumberPaginator

config: RESTAPIConfig = {
    "client": {
        "base_url": "https://api.github.com",
        "auth": {
            "token": dlt.secrets["sources.access_token"],
        },
        "paginator": "header_link",
    },
    "resources": [
        {
            "name": "contributors",
            "endpoint": {
                "path": "repos/dlt-hub/dlt/contributors",
            },
        },
    ],
}

github_source = rest_api_source(config)


rest_api_pipeline = dlt.pipeline(
    pipeline_name="rest_api_github",
    destination="duckdb",
    dataset_name="rest_api_data",
    dev_mode=True,
)

load_info = rest_api_pipeline.run(github_source)
print(load_info)

Pipeline rest_api_github load step finished in 0.53 seconds
1 load package(s) were loaded to destination duckdb and into dataset rest_api_data_20261120035117
The duckdb destination used duckdb:////content/rest_api_github.duckdb location to store data
Load package 1795189877.1010094 is LOADED and contains no failed jobs


In [39]:
rest_api_pipeline.dataset().contributors.df()

,login,id,node_id,avatar_url,gravatar_id,url,html_url,followers_url,following_url,gists_url,...,organizations_url,repos_url,events_url,received_events_url,type,user_view_type,site_admin,contributions,_dlt_load_id,_dlt_id
0,rudolfix,17202864,MDQ6VXNlcjE3MjAyODY0,https://avatars.githubusercontent.com/u/172028...,,https://api.github.com/users/rudolfix,https://github.com/rudolfix,https://api.github.com/users/rudolfix/followers,https://api.github.com/users/rudolfix/followin...,https://api.github.com/users/rudolfix/gists{/g...,...,https://api.github.com/users/rudolfix/orgs,https://api.github.com/users/rudolfix/repos,https://api.github.com/users/rudolfix/events{/...,https://api.github.com/users/rudolfix/received...,User,public,False,1638,1795189877.1010094,GicjX6fwEmA2EQ
1,sh-rp,1155738,MDQ6VXNlcjExNTU3Mzg=,https://avatars.githubusercontent.com/u/115573...,,https://api.github.com/users/sh-rp,https://github.com/sh-rp,https://api.github.com/users/sh-rp/followers,https://api.github.com/users/sh-rp/following{/...,https://api.github.com/users/sh-rp/gists{/gist...,...,https://api.github.com/users/sh-rp/orgs,https://api.github.com/users/sh-rp/repos,https://api.github.com/users/sh-rp/events{/pri...,https://api.github.com/users/sh-rp/received_ev...,User,public,False,383,1795189877.1010094,17dSbbGWn8pNPg
2,burnash,264674,MDQ6VXNlcjI2NDY3NA==,https://avatars.githubusercontent.com/u/264674...,,https://api.github.com/users/burnash,https://github.com/burnash,https://api.github.com/users/burnash/followers,https://api.github.com/users/burnash/following...,https://api.github.com/users/burnash/gists{/gi...,...,https://api.github.com/users/burnash/orgs,https://api.github.com/users/burnash/repos,https://api.github.com/users/burnash/events{/p...,https://api.github.com/users/burnash/received_...,User,public,False,292,1795189877.1010094,k7w6v6nwG+84DA
3,steinitzu,1033963,MDQ6VXNlcjEwMzM5NjM=,https://avatars.githubusercontent.com/u/103396...,,https://api.github.com/users/steinitzu,https://github.com/steinitzu,https://api.github.com/users/steinitzu/followers,https://api.github.com/users/steinitzu/followi...,https://api.github.com/users/steinitzu/gists{/...,...,https://api.github.com/users/steinitzu/orgs,https://api.github.com/users/steinitzu/repos,https://api.github.com/users/steinitzu/events{...,https://api.github.com/users/steinitzu/receive...,User,public,False,255,1795189877.1010094,9vHSCvEhkyTyOQ
4,TyDunn,13314504,MDQ6VXNlcjEzMzE0NTA0,https://avatars.githubusercontent.com/u/133145...,,https://api.github.com/users/TyDunn,https://github.com/TyDunn,https://api.github.com/users/TyDunn/followers,https://api.github.com/users/TyDunn/following{...,https://api.github.com/users/TyDunn/gists{/gis...,...,https://api.github.com/users/TyDunn/orgs,https://api.github.com/users/TyDunn/repos,https://api.github.com/users/TyDunn/events{/pr...,https://api.github.com/users/TyDunn/received_e...,User,public,False,252,1795189877.1010094,zBDwDHPofnBB7Q
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,jonbiemond,101326085,U_kgDOBgodBQ,https://avatars.githubusercontent.com/u/101326...,,https://api.github.com/users/jonbiemond,https://github.com/jonbiemond,https://api.github.com/users/jonbiemond/followers,https://api.github.com/users/jonbiemond/follow...,https://api.github.com/users/jonbiemond/gists{...,...,https://api.github.com/users/jonbiemond/orgs,https://api.github.com/users/jonbiemond/repos,https://api.github.com/users/jonbiemond/events...,https://api.github.com/users/jonbiemond/receiv...,User,public,False,1,1795189877.1010094,BmuUcQT/OW5jnw
214,vincere-mori,57835745,MDQ6VXNlcjU3ODM1NzQ1,https://avatars.githubusercontent.com/u/578357...,,https://api.github.com/users/vincere-mori,https://github.com/vincere-mori,https://api.github.com/users/vincere-mori/foll...,https://api.github.com/users/vincere-mori/foll...,https://api.github.com/users/vincere-mori/gist...,...,https://api.github.com/users/vincere-mori/orgs,https://api.github.com/users

---
## **[SQL Databases source](https://dlthub.com/docs/dlt-ecosystem/verified-sources/sql_database/)**

SQL databases are management systems (DBMS) that store data in a structured format, commonly used for efficient and reliable data retrieval.

The `sql_database` verified source loads data to your specified destination using one of the following backends:
* SQLAlchemy,
* PyArrow,
* pandas,
* ConnectorX.

### Initialize the `sql_database` template

Initialize the `dlt` template for `sql_database` using the `init` command:


In [40]:
!yes | dlt init sql_database duckdb

Creating a new pipeline with the dlt core source sql_database (Source that loads tables form any SQLAlchemy supported database, supports batching requests and incremental loads.)
NOTE: Beginning with dlt 1.0.0, the source sql_database will no longer be copied from the verified sources repo but imported from dlt.sources. You can provide the --eject flag to revert to the old behavior.

Your new pipeline sql_database is ready to be customized!
* Review and change how dlt loads your data in sql_database_pipeline.py
* Add credentials for duckdb and other secrets to ./.dlt/secrets.toml
* Add the required dependencies to requirements.txt:
  dlt[duckdb,sql-database]>=1.32.0
  To install with pip: pip3 install 'dlt[duckdb,sql-database]>=1.32.0'

* Read https://dlthub.com/docs/walkthroughs/create-a-pipeline for more information


The `sql_database` source is also a **built-in source**, you don't have to initialize it, just **import** it from `dlt.sources`.

### Example

The example below shows how you can use dlt to load data from a SQL database (PostgreSQL, MySQL, SQLite, Oracle, IBM DB2, etc.) into a destination.

To make it easy to reproduce, we will load data from the [public MySQL Rfam database](https://docs.rfam.org/en/latest/database.html) into a local DuckDB instance.

In [41]:
%%capture
!pip install pymysql

In [42]:
from dlt.sources.sql_database import sql_database

sql_source = sql_database(
    "mysql+pymysql://rfamro@mysql-rfam-public.ebi.ac.uk:4497/Rfam",
    table_names=[
        "family",
    ],
)

sql_db_pipeline = dlt.pipeline(
    pipeline_name="sql_database_example",
    destination="duckdb",
    dataset_name="sql_data",
    dev_mode=True,
)

load_info = sql_db_pipeline.run(sql_source)
print(load_info)

2026-11-20 15:52:54,396|[WARNING]|1204|137020450766976|dlt|schema_types.py|sqla_col_to_column_schema:199|A column with name gathering_cutoff contains unknown data type DOUBLE which cannot be mapped to `dlt` data type. When using sqlalchemy backend such data will be passed to the normalizer. In case of `pyarrow` and `pandas` backend, data types are detected from numpy ndarrays. In case of other backends, the behavior is backend-specific.
2026-11-20 15:52:54,399|[WARNING]|1204|137020450766976|dlt|schema_types.py|sqla_col_to_column_schema:199|A column with name trusted_cutoff contains unknown data type DOUBLE which cannot be mapped to `dlt` data type. When using sqlalchemy backend such data will be passed to the normalizer. In case of `pyarrow` and `pandas` backend, data types are detected from numpy ndarrays. In case of other backends, the behavior is backend-specific.
2026-11-20 15:52:54,401|[WARNING]|1204|137020450766976|dlt|schema_types.py|sqla_col_to_column_schema:199|A column with n

Pipeline sql_database_example load step finished in 2.16 seconds
1 load package(s) were loaded to destination duckdb and into dataset sql_data_20261120035254
The duckdb destination used duckdb:////content/sql_database_example.duckdb location to store data
Load package 1795189974.8131728 is LOADED and contains no failed jobs


### **Exercise 3: Run `sql_database` source**

Explore the cells above and answer the question below using `sql_client` or `pipeline.dataset()`.

#### **Question**
How many columns does the `family` table have?

In [46]:
sql_db_pipeline.dataset().family.df()

,rfam_acc,rfam_id,auto_wiki,description,author,seed_source,comment,previous_id,cmbuild,cmcalibrate,...,updated,gathering_cutoff,trusted_cutoff,noise_cutoff,ecmli_lambda,ecmli_mu,hmm_tau,hmm_lambda,_dlt_load_id,_dlt_id
0,RF00001,5S_rRNA,1302,5S ribosomal RNA,"Griffiths-Jones SR, Mifsud W, Gardner PP","Szymanski et al, 5S ribosomal database, PMID:1...",5S ribosomal RNA (5S rRNA) is a component of t...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-09 21:15:12,38.0,38.0,37.9,0.60889,-5.17319,-3.7679,0.71820,1795189974.8131728,r9+qnmJ+FEtOQw
1,RF00002,5_8S_rRNA,1303,5.8S ribosomal RNA,"Griffiths-Jones SR, Mifsud W","Wuyts et al, European LSU rRNA database, PMID:...",5.8S ribosomal RNA (5.8S rRNA) is a component ...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-10 04:51:11,42.0,42.0,41.9,0.66150,-9.18781,-3.4977,0.71788,1795189974.8131728,Ol+mXzqWnZndHQ
2,RF00003,U1,1304,U1 spliceosomal RNA,"Griffiths-Jones SR, Mifsud W, Moxon SJ, Ontive...","Zwieb C, The uRNA database, PMID:9016512",U1 is a small nuclear RNA (snRNA) component of...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 14:57:22,69.5,69.5,69.4,0.66474,-8.65259,-3.7311,0.71618,1795189974.8131728,G4qUVdr8hgNz4Q
3,RF00004,U2,1305,U2 spliceosomal RNA,"Griffiths-Jones SR, Mifsud W, Gardner PP","The uRNA database, PMID:9016512; Griffiths-Jon...",U2 is a small nuclear RNA (snRNA) component of...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-10 04:51:11,46.0,46.0,45.9,0.55201,-9.92571,-3.4862,0.71304,1795189974.8131728,qS4RGnEAytXfTw
4,RF00005,tRNA,1306,tRNA,"Eddy SR, Griffiths-Jones SR, Mifsud W",Eddy SR,Transfer RNA (tRNA) molecules are approximatel...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-10 04:51:11,29.0,29.0,28.9,0.63376,-4.41829,-2.6284,0.73562,1795189974.8131728,L4THoti/mjr7ag
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4222,RF04356,Plastid-ndhA-ndhB-1,2777,Plastid-ndhA-ndhB-1 group II-D5 intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid ndhA-ndhB-1 is a non-coding RNA (ncRNA...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,61.0,61.0,60.9,0.69637,-3.94316,-3.9178,0.71944,1795189974.8131728,f614/K5hlGMeTQ
4223,RF04357,Plastid-ndhA-ndhB-2,2777,Plastid-ndhA-ndhB-2 group II-D5 intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid ndhA-ndhB-2 is a non-coding RNA (ncRNA...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,53.0,53.0,52.9,0.65759,-5.53104,-3.8375,0.71961,1795189974.8131728,cuQ60lche6xvhg
4224,RF04358,Plastid-ndhA-ndhB-3,2777,Plastid-ndhA-ndhB-3 group II-D5 intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid ndhA-ndhB-3 is a non-coding RNA (ncRNA...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,90.0,90.0,89.9,0.41590,-7.91967,-2.9394,0.71546,1795189974.8131728,pj/8LjEj0U5iKg
4225,RF04359,Plastid-GIID1-clpP,2777,Plastid group II-D1 clpP intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid-GIID1-clpP is a non-coding RNA (ncRNA)...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,140.0,140.0,139.8,0.53772,-8.24418,-2.9374,0.71444,1795189974.8131728,eXqgVwAqS20/sg


---
## **[Filesystem source](https://dlthub.com/docs/dlt-ecosystem/verified-sources/filesystem/)**

The filesystem source allows seamless loading of files from the following locations:

* AWS S3
* Google Cloud Storage
* Google Drive
* Azure Blob Storage
* remote filesystem (via SFTP)
* local filesystem

The filesystem source natively supports CSV, Parquet, and JSONL files and allows customization for loading any type of structured file.


**How filesystem source works**

The Filesystem source doesn't just give you an easy way to load data from both remote and local files — it also comes with a powerful set of tools that let you customize the loading process to fit your specific needs.

Filesystem source loads data in two steps:

1. It accesses the files in your remote or local file storage **without** actually **reading** the content yet. At this point, you can filter files by metadata or name. You can also set up incremental loading to load only new files.
2. The **transformer** **reads** the files' content and yields the records. At this step, you can filter out the actual data, enrich records with metadata from files, or perform incremental loading based on the file content.

### Initialize the `filesystem` template

Initialize the dlt template for `filesystem` using the `init` command:


In [47]:
!yes | dlt init filesystem duckdb

Creating a new pipeline with the dlt core source filesystem (Reads files in s3, gs or azure buckets using fsspec and provides convenience resources for chunked reading of various file formats)
NOTE: Beginning with dlt 1.0.0, the source filesystem will no longer be copied from the verified sources repo but imported from dlt.sources. You can provide the --eject flag to revert to the old behavior.

Your new pipeline filesystem is ready to be customized!
* Review and change how dlt loads your data in filesystem_pipeline.py
* Add credentials for duckdb and other secrets to ./.dlt/secrets.toml
* Add the required dependencies to requirements.txt:
  dlt[duckdb,filesystem]>=1.32.0
  To install with pip: pip3 install 'dlt[duckdb,filesystem]>=1.32.0'

* Read https://dlthub.com/docs/walkthroughs/create-a-pipeline for more information


The `filesystem` source is also a **built-in source**, you don't have to initialize it, just **import** it from `dlt.sources`.

### Example

To illustrate how this **built-in source** works, we first download some file to the local (Colab) filesystem.

In [48]:
import os
import requests

folder_name = "local_data"
os.makedirs(folder_name, exist_ok=True)
full_path = os.path.abspath(folder_name)

url = "https://www.timestored.com/data/sample/userdata.parquet"
resp = requests.get(url)
resp.raise_for_status()

with open(f"{full_path}/userdata.parquet", "wb") as f:
    f.write(resp.content)

In [49]:
import dlt
from dlt.sources.filesystem import filesystem, read_parquet

filesystem_resource = filesystem(bucket_url=full_path, file_glob="**/*.parquet")
filesystem_pipe = filesystem_resource | read_parquet()

# We load the data into the table_name table
fs_pipeline = dlt.pipeline(pipeline_name="my_pipeline", destination="duckdb")
load_info = fs_pipeline.run(filesystem_pipe.with_name("userdata"))
print(load_info)

Pipeline my_pipeline load step finished in 0.43 seconds
1 load package(s) were loaded to destination duckdb and into dataset my_pipeline_dataset
The duckdb destination used duckdb:////content/my_pipeline.duckdb location to store data
Load package 1795190107.735487 is LOADED and contains no failed jobs


### **Exercise 4: Run `filesystem` source**

Explore the cells above and answer the question below using `sql_client` or `pipeline.dataset()`.

#### **Question**
How many columns does the `userdata` table have?


In [51]:
fs_pipeline.dataset().userdata.df()

,registration_dttm,id,first_name,last_name,email,gender,ip_address,cc,country,birthdate,salary,title,comments,_dlt_load_id,_dlt_id
0,2016-02-03 07:55:29+00:00,1,Amanda,Jordan,ajordan0@com.com,Female,1.197.201.2,6759521864920116,Indonesia,3/8/1971,49756.53,Internal Auditor,1E+02,1795190107.735487,9rWqkoT8wCQhmg
1,2016-02-03 17:04:03+00:00,2,Albert,Freeman,afreeman1@is.gd,Male,218.111.175.34,,Canada,1/16/1968,150280.17,Accountant IV,,1795190107.735487,94+0N8gE3dZjyQ
2,2016-02-03 01:09:31+00:00,3,Evelyn,Morgan,emorgan2@altervista.org,Female,7.161.136.94,6767119071901597,Russia,2/1/1960,144972.51,Structural Engineer,,1795190107.735487,pRlo/Y8+TBZxSA
3,2016-02-03 00:36:21+00:00,4,Denise,Riley,driley3@gmpg.org,Female,140.35.109.83,3576031598965625,China,4/8/1997,90263.05,Senior Cost Accountant,,1795190107.735487,4nj+mF/gKnB1FA
4,2016-02-03 05:05:31+00:00,5,Carlos,Burns,cburns4@miitbeian.gov.cn,,169.113.235.40,5602256255204850,South Africa,,NaN,,,1795190107.735487,WsjYYdpocPiMcg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2016-02-03 10:30:59+00:00,996,Dennis,Harris,dharrisrn@eepurl.com,Male,178.180.111.236,374288806662929,Greece,7/8/1965,263399.54,Editor,,1795190107.735487,7Si/lZjQI7RmRQ
996,2016-02-03 17:16:53+00:00,997,Gloria,Hamilton,ghamiltonro@rambler.ru,Female,71.50.39.137,,China,4/22/1975,83183.54,VP Product Management,,1795190107.735487,01WOjzPITtmdcg
997,2016-02-03 05:02:20+00:00,998,Nancy,Morris,nmorrisrp@ask.com,,6.188.121.221,3553564071014997,Sweden,5/1/1979,NaN,Junior Executive,,1795190107.735487,V1N753Wz2o5log
998,2016-02-03 02:41:32+00:00,999,Annie,Daniels,adanielsrq@squidoo.com,Female,97.221.132.35,30424803513734,China,10/9/1991,18433.85,Editor,​,1795190107.735487,ZUxgw9jb23ER6g


You can read how to configure **Cloud Storage** in the official
[dlt documentation](https://dlthub.com/docs/dlt-ecosystem/verified-sources/filesystem#configuration).


# [**Built-in Destinations**](https://dlthub.com/docs/dlt-ecosystem/destinations/)


![Lesson_4_Using_pre_build_sources_and_destinations_img2](https://storage.googleapis.com/dlt-blog-images/dlt-fundamentals-course/Lesson_4_Using_pre_build_sources_and_destinations_img2.png)

---
##  **Exploring `dlt` destinations**


To be honest, this is simply a matter of going through the
[documentation](https://dlthub.com/docs/dlt-ecosystem/destinations/) 👀, but to sum it up:

- Most likely, the destination where you want to load data is already a `dlt` integration that undergoes several hundred automated tests every day.
- If not, you can define a custom destination and still benefit from most `dlt`-specific features.
  *FYI: custom destinations will be covered in the next Advanced course — so we expect you to come back for part two…*


## **Choosing a destination**

Switching between destinations in `dlt` is incredibly straightforward. Simply modify the `destination` parameter in your pipeline configuration. For example:

In [52]:
data_pipeline = dlt.pipeline(
    pipeline_name="data_pipeline",
    destination="duckdb",
    dataset_name="data",
)
print(data_pipeline.destination.destination_type)

data_pipeline = dlt.pipeline(
    pipeline_name="data_pipeline",
    destination="bigquery",
    dataset_name="data",
)
print(data_pipeline.destination.destination_type)

2026-11-20 15:56:20,760|[WARNING]|1204|137020450766976|dlt|pipeline.py|_state_to_props:2064|The destination dlt.destinations.duckdb:None in state differs from destination dlt.destinations.bigquery:bigquery in pipeline and will be ignored


dlt.destinations.duckdb
dlt.destinations.bigquery


This flexibility allows you to easily transition from local development to production-grade environments.



## **Filesystem destination**

The `filesystem` destination enables you to load data into **files stored locally** or in **cloud storage** solutions, making it an excellent choice for lightweight testing, prototyping, or file-based workflows.

Below is an **example** demonstrating how to use the `filesystem` destination to load data in **Parquet** format:

* Step 1: Set up a local bucket or cloud directory for storing files


In [53]:
import os

os.environ["BUCKET_URL"] = "./content"

* Step 2: Define the data source

In [54]:
import dlt
from dlt.sources.sql_database import sql_database

source = sql_database(
    "mysql+pymysql://rfamro@mysql-rfam-public.ebi.ac.uk:4497/Rfam",
    table_names=[
        "family",
    ],
)


pipeline = dlt.pipeline(
    pipeline_name="fs_pipeline",
    destination="filesystem",
    dataset_name="fs_data",
)

load_info = pipeline.run(source, loader_file_format="parquet")
print(load_info)

2026-11-20 15:56:29,805|[WARNING]|1204|137020450766976|dlt|schema_types.py|sqla_col_to_column_schema:199|A column with name gathering_cutoff contains unknown data type DOUBLE which cannot be mapped to `dlt` data type. When using sqlalchemy backend such data will be passed to the normalizer. In case of `pyarrow` and `pandas` backend, data types are detected from numpy ndarrays. In case of other backends, the behavior is backend-specific.
2026-11-20 15:56:29,806|[WARNING]|1204|137020450766976|dlt|schema_types.py|sqla_col_to_column_schema:199|A column with name trusted_cutoff contains unknown data type DOUBLE which cannot be mapped to `dlt` data type. When using sqlalchemy backend such data will be passed to the normalizer. In case of `pyarrow` and `pandas` backend, data types are detected from numpy ndarrays. In case of other backends, the behavior is backend-specific.
2026-11-20 15:56:29,807|[WARNING]|1204|137020450766976|dlt|schema_types.py|sqla_col_to_column_schema:199|A column with n

Pipeline fs_pipeline load step finished in 0.02 seconds
1 load package(s) were loaded to destination filesystem and into dataset fs_data
The filesystem destination used file:///content/content location to store data
Load package 1795190189.947674 is LOADED and contains no failed jobs


Look at the files:

In [55]:
! ls ./content/fs_data/family

1795190189.947674.15d4bfbe87.parquet


Look at the loaded data:

In [56]:
# explore loaded data
pipeline.dataset().family.df()

,rfam_acc,rfam_id,auto_wiki,description,author,seed_source,comment,previous_id,cmbuild,cmcalibrate,...,updated,gathering_cutoff,trusted_cutoff,noise_cutoff,ecmli_lambda,ecmli_mu,hmm_tau,hmm_lambda,_dlt_load_id,_dlt_id
0,RF00001,5S_rRNA,1302,5S ribosomal RNA,"Griffiths-Jones SR, Mifsud W, Gardner PP","Szymanski et al, 5S ribosomal database, PMID:1...",5S ribosomal RNA (5S rRNA) is a component of t...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-09 21:15:12,38.0,38.0,37.9,0.60889,-5.17319,-3.7679,0.71820,1795190189.947674,wLzmW+2MfS9Odw
1,RF00002,5_8S_rRNA,1303,5.8S ribosomal RNA,"Griffiths-Jones SR, Mifsud W","Wuyts et al, European LSU rRNA database, PMID:...",5.8S ribosomal RNA (5.8S rRNA) is a component ...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-10 04:51:11,42.0,42.0,41.9,0.66150,-9.18781,-3.4977,0.71788,1795190189.947674,C7jii1+6r3UK2Q
2,RF00003,U1,1304,U1 spliceosomal RNA,"Griffiths-Jones SR, Mifsud W, Moxon SJ, Ontive...","Zwieb C, The uRNA database, PMID:9016512",U1 is a small nuclear RNA (snRNA) component of...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 14:57:22,69.5,69.5,69.4,0.66474,-8.65259,-3.7311,0.71618,1795190189.947674,qarbV3Hv2MsBeQ
3,RF00004,U2,1305,U2 spliceosomal RNA,"Griffiths-Jones SR, Mifsud W, Gardner PP","The uRNA database, PMID:9016512; Griffiths-Jon...",U2 is a small nuclear RNA (snRNA) component of...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-10 04:51:11,46.0,46.0,45.9,0.55201,-9.92571,-3.4862,0.71304,1795190189.947674,nGC/4doIk8k9Ow
4,RF00005,tRNA,1306,tRNA,"Eddy SR, Griffiths-Jones SR, Mifsud W",Eddy SR,Transfer RNA (tRNA) molecules are approximatel...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2024-09-10 04:51:11,29.0,29.0,28.9,0.63376,-4.41829,-2.6284,0.73562,1795190189.947674,l8quyaISdPxn+w
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4222,RF04356,Plastid-ndhA-ndhB-1,2777,Plastid-ndhA-ndhB-1 group II-D5 intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid ndhA-ndhB-1 is a non-coding RNA (ncRNA...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,61.0,61.0,60.9,0.69637,-3.94316,-3.9178,0.71944,1795190189.947674,NqXm/fu66jNqPg
4223,RF04357,Plastid-ndhA-ndhB-2,2777,Plastid-ndhA-ndhB-2 group II-D5 intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid ndhA-ndhB-2 is a non-coding RNA (ncRNA...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,53.0,53.0,52.9,0.65759,-5.53104,-3.8375,0.71961,1795190189.947674,Prl/pzY31uUzRw
4224,RF04358,Plastid-ndhA-ndhB-3,2777,Plastid-ndhA-ndhB-3 group II-D5 intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid ndhA-ndhB-3 is a non-coding RNA (ncRNA...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,90.0,90.0,89.9,0.41590,-7.91967,-2.9394,0.71546,1795190189.947674,fEW3eD1LbF8K6Q
4225,RF04359,Plastid-GIID1-clpP,2777,Plastid group II-D1 clpP intron,"Mehta D, Reis R, Hua J",Mehta D,Plastid-GIID1-clpP is a non-coding RNA (ncRNA)...,None,cmbuild -F CM SEED,cmcalibrate --mpi CM,...,2025-12-11 15:52:40,140.0,140.0,139.8,0.53772,-8.24418,-2.9374,0.71444,1795190189.947674,nNOaZ8mIIuhGZg


### **Table formats: [Delta tables & Iceberg](https://dlthub.com/docs/dlt-ecosystem/destinations/delta-iceberg)**

dlt supports writing **Delta** and **Iceberg** tables when using the `filesystem` destination.

**How it works:**

dlt uses the `deltalake` and `pyiceberg` libraries to write Delta and Iceberg tables, respectively. One or multiple Parquet files are prepared during the extract and normalize steps. In the load step, these Parquet files are exposed as an Arrow data structure and fed into `deltalake` or `pyiceberg`.

In [57]:
%%capture
!pip install "dlt[pyiceberg]"

In [58]:
load_info = pipeline.run(
    source,
    loader_file_format="parquet",
    table_format="iceberg",
)
print(load_info)

Pipeline fs_pipeline load step finished in 4.97 seconds
1 load package(s) were loaded to destination filesystem and into dataset fs_data
The filesystem destination used file:///content/content location to store data
Load package 1795190213.1514378 is LOADED and contains no failed jobs


**Note:**

The open-source version of dlt supports basic functionality for **Iceberg**. A more powerful Iceberg integration is available as part of **dltHub**: the [Iceberg destination](https://dlthub.com/docs/hub/ingestion/iceberg) loads data into Iceberg tables using the `pyiceberg` library, and supports multiple catalog types and both local and cloud storage backends.

# **Spoiler: Custom Sources & Destinations**

`dlt` aims to simplify the process of creating both custom sources
([REST API Client](https://dlthub.com/docs/general-usage/http/rest-client),
[`rest_api` source](https://dlthub.com/docs/dlt-ecosystem/verified-sources/rest_api))
and [custom destinations](https://dlthub.com/docs/dlt-ecosystem/destinations/destination).

We will explore this topic in more detail in the next Advanced course.


✅ ▶ Proceed to the [next lesson](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_5_write_disposition_and_incremental_loading.ipynb)!